# CPC353 Assignment 1 — N-gram Analysis

This notebook analyses bigram and trigram patterns in the combined instant noodle review corpus. The 2026-11-09 snapshot contains all 125 review rows (57 collected, 68 AI-generated) across products; no contributor, ID-prefix or language filter is applied.

Run `preprocessing.ipynb` first. The notebook saves analysis tables in `data/analysis/ngram/`.

**Research context:** N-gram analysis helps identify frequently co-occurring word pairs and triples that reveal product-attribute discussions and code-switching patterns. Cross-language bigrams (English–Malay adjacencies) are particularly informative for understanding how reviewers switch between languages within phrases.

## 1. Imports and locate the project folder

The notebook searches the current folder and its parents for the preprocessing outputs, so it works when launched from the repository root or a subfolder.

In [1]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

PROJECT_DIR = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "data" / "processed").is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Could not find data/processed. Run this notebook from the project.")

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "data" / "analysis" / "ngram"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from IPython.display import display
except ImportError:
    display = print

print(f"Project folder: {PROJECT_DIR}")
print(f"N-gram outputs: {OUTPUT_DIR}")

Project folder: c:\Users\Admin\OneDrive\Documents\GitHub\CPC353-Assignment-1
N-gram outputs: c:\Users\Admin\OneDrive\Documents\GitHub\CPC353-Assignment-1\data\analysis\ngram


## 2. Load the retained reviews and project stop words

All included collected and AI-generated reviews are combined into one corpus. We load the project stop-word list to produce both unfiltered and filtered n-gram views.

In [2]:
def load_jsonl(path):
    with path.open(encoding="utf-8") as file:
        rows = [json.loads(line) for line in file if line.strip()]
    return [row for row in rows if row.get("included", True)]

reviews = (
    load_jsonl(PROCESSED_DIR / "real_preprocessed.jsonl")
    + load_jsonl(PROCESSED_DIR / "ai_preprocessed.jsonl")
)

stopwords_df = pd.read_csv(PROCESSED_DIR / "stopwords_used.csv")
stopwords_df["word"] = stopwords_df["word"].astype(str).str.casefold()
stopword_set = set(stopwords_df["word"])

print(f"Included reviews in combined corpus: {len(reviews)}")
print(f"Project stop words loaded: {len(stopword_set)}")

Included reviews in combined corpus: 125
Project stop words loaded: 100


## 3. Build a bilingual word-language lookup

To identify cross-language n-grams, we build a simple word-to-language lookup from the project stop-word list, protected words, and the concordance lexicon. This uses the project's own classifications, not automatic language detection. Words not in the lookup are labelled by a simple heuristic based on common Malay word patterns; ambiguous cases are marked `ambiguous`.

In [3]:
import csv

word_lang = {}
for _, row in stopwords_df.iterrows():
    word_lang[row["word"]] = row["language"]

protected_df = pd.read_csv(PROCESSED_DIR / "protected_words.csv")
for _, row in protected_df.iterrows():
    w = str(row["word"]).casefold()
    if w not in word_lang and "reason" in row.index:
        reason = str(row["reason"]).lower()
        if "malay" in reason or "bahasa" in reason:
            word_lang[w] = "Malay"
        elif "english" in reason:
            word_lang[w] = "English"

lexicon_path = PROJECT_DIR / "config" / "concordance_lexicon.csv"
if lexicon_path.exists():
    with lexicon_path.open(encoding="utf-8-sig") as f:
        for entry in csv.DictReader(f):
            expr = entry["expression"].casefold().strip()
            lang = entry.get("language", "").strip()
            if " " not in expr:  # single words only for token lookup
                if "malay" in lang.lower():
                    word_lang[expr] = "Malay"
                elif "english" in lang.lower():
                    word_lang[expr] = "English"

KNOWN_MALAY = {
    "sedap", "pedas", "kuah", "kuahnya", "kenyal", "rasa", "mi", "kari",
    "lembik", "masin", "mahal", "harga", "bungkusan", "masak", "rebus",
    "goreng", "makan", "suka", "sangat", "memang", "jugak", "perlu",
    "sesuai", "tekak", "makanan", "berbeza", "buat", "celur", "tambah",
    "masih", "agak", "cair", "pekat", "rangup", "lemak", "panas",
    "enak", "nikmat", "lazat", "sedikit", "banyak", "topping",
    "gila", "punya", "bila", "boleh", "ganti", "siapa", "pandai",
    "tengok", "letak", "pon", "mcm", "jela", "mmg", "sgt",
    "kaw", "firma", "kalau", "jadi", "dekat", "sebab", "lagi",
}
KNOWN_ENGLISH = {
    "noodles", "noodle", "instant", "spicy", "taste", "flavour", "flavor",
    "broth", "soup", "texture", "springy", "chewy", "salty", "crispy",
    "price", "pricey", "packaging", "packet", "portion", "good", "best",
    "love", "like", "enjoy", "recommend", "worth", "cooking", "cook",
    "rich", "bold", "satisfying", "delicious", "amazing", "perfect",
    "firm", "soft", "soggy", "bland", "thick", "thin", "hot",
    "spice", "heat", "kick", "seasoning", "condiment", "flakes",
    "favourite", "favorite", "definitely", "really", "quite",
}
BRAND_TOKENS = {"shin", "ramyun", "ramyum", "nongshim", "nongshim's", "maggi",
                "korea", "korean", "malaysia", "malaysian"}

for w in KNOWN_MALAY:
    if w not in word_lang:
        word_lang[w] = "Malay"
for w in KNOWN_ENGLISH:
    if w not in word_lang:
        word_lang[w] = "English"

def get_token_language(token):
    """Return the language label for a token, or 'ambiguous' if unknown."""
    t = token.casefold()
    if t in BRAND_TOKENS:
        return "brand"
    return word_lang.get(t, "ambiguous")

print(f"Word-language lookup entries: {len(word_lang)}")
print(f"Known Malay content words: {len(KNOWN_MALAY)}")
print(f"Known English content words: {len(KNOWN_ENGLISH)}")

Word-language lookup entries: 217
Known Malay content words: 63
Known English content words: 52


## 4. Generate n-grams from unfiltered sentence tokens

N-grams are extracted within sentence boundaries to avoid spurious cross-sentence pairs. We use `tokens_before` (unfiltered) so that function words and code-switching markers are preserved. Stop-word-filtered n-grams are computed separately for content-word patterns.

In [4]:
def extract_ngrams(tokens, n):
    """Extract n-grams from a token list."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def corpus_ngrams(reviews, n, use_field="sentence_tokens_before", filter_stopwords=False):
    """Extract n-grams within sentence boundaries across the full corpus.
    
    Returns Counter of n-gram tuples and a dict mapping n-gram -> set of (origin, review_id).
    """
    counts = Counter()
    review_sets = {}
    for row in reviews:
        if row.get("included") is False:
            continue
        key = (row["origin"], row["review_id"])
        for sentence in row.get(use_field, []):
            tokens = [t.casefold() for t in sentence]
            if filter_stopwords:
                tokens = [t for t in tokens if t not in stopword_set]
            grams = extract_ngrams(tokens, n)
            for gram in grams:
                counts[gram] += 1
                review_sets.setdefault(gram, set()).add(key)
    return counts, review_sets

bigram_counts, bigram_reviews = corpus_ngrams(reviews, 2)
trigram_counts, trigram_reviews = corpus_ngrams(reviews, 3)

bigram_filtered, bigram_filt_reviews = corpus_ngrams(reviews, 2, filter_stopwords=True)
trigram_filtered, trigram_filt_reviews = corpus_ngrams(reviews, 3, filter_stopwords=True)

total_unfiltered = sum(len(row["tokens_before"]) for row in reviews if row.get("included") is not False)

print(f"Unfiltered bigram types: {len(bigram_counts):,}")
print(f"Unfiltered trigram types: {len(trigram_counts):,}")
print(f"Filtered bigram types (stop words removed): {len(bigram_filtered):,}")
print(f"Filtered trigram types (stop words removed): {len(trigram_filtered):,}")
print(f"Total unfiltered tokens: {total_unfiltered:,}")

Unfiltered bigram types: 1,666
Unfiltered trigram types: 1,761
Filtered bigram types (stop words removed): 1,259
Filtered trigram types (stop words removed): 1,185
Total unfiltered tokens: 2,252


## 5. Most frequent bigrams

The tables below show the top 25 bigrams by frequency, both before and after stop-word filtering. Unfiltered bigrams reveal grammatical and structural patterns, while filtered bigrams highlight content-word co-occurrences related to product attributes.

In [5]:
def ngram_to_df(counts, review_sets, total_tokens, top_n=25, label="bigram"):
    """Convert n-gram counts to a DataFrame with frequency and review coverage."""
    rows = []
    for gram, count in counts.most_common(top_n):
        gram_str = " ".join(gram)
        langs = [get_token_language(t) for t in gram]
        lang_pair = " + ".join(langs)
        # Determine if cross-language
        real_langs = {l for l in langs if l not in ("brand", "ambiguous")}
        is_cross = len(real_langs) > 1
        rows.append({
            "ngram": gram_str,
            "frequency": count,
            "reviews_containing": len(review_sets.get(gram, set())),
            "per_1000_tokens": round(count / total_tokens * 1000, 3) if total_tokens else 0,
            "token_languages": lang_pair,
            "cross_language": is_cross,
        })
    return pd.DataFrame(rows)

print("=" * 70)
print("TOP 25 UNFILTERED BIGRAMS")
print("=" * 70)
bigram_df = ngram_to_df(bigram_counts, bigram_reviews, total_unfiltered, 25)
display(bigram_df)

print("\n" + "=" * 70)
print("TOP 25 FILTERED BIGRAMS (stop words removed)")
print("=" * 70)
bigram_filt_df = ngram_to_df(bigram_filtered, bigram_filt_reviews, total_unfiltered, 25)
display(bigram_filt_df)

TOP 25 UNFILTERED BIGRAMS


,ngram,frequency,reviews_containing,per_1000_tokens,token_languages,cross_language
0,shin ramyun,57,57,25.311,brand + brand,False
1,the noodles,17,17,7.549,English + English,False
2,and the,11,11,4.885,English + English,False
3,but the,9,9,3.996,ambiguous + English,False
4,noodles are,7,6,3.108,English + English,False
5,instant noodles,7,7,3.108,English + English,False
6,saya suka,7,7,3.108,Malay + Malay,False
7,the flavour,6,6,2.664,English + English,False
8,of the,6,6,2.664,English + English,False
9,kenyal dan,6,6,2.664,Malay + Malay,False



TOP 25 FILTERED BIGRAMS (stop words removed)


,ngram,frequency,reviews_containing,per_1000_tokens,token_languages,cross_language
0,shin ramyun,57,57,25.311,brand + brand,False
1,instant noodles,7,7,3.108,English + English,False
2,mi shin,6,6,2.664,Malay + brand,False
3,maggi kari,5,4,2.220,brand + Malay,False
4,ramyun punya,5,5,2.220,brand + Malay,False
5,kenyal kuahnya,5,5,2.220,Malay + Malay,False
6,kuah shin,4,4,1.776,Malay + brand,False
7,ramyun sedap,4,4,1.776,brand + Malay,False
8,one packet,4,4,1.776,ambiguous + English,False
9,not too,3,2,1.332,ambiguous + ambiguous,False


## 6. Most frequent trigrams

Trigrams capture longer patterns such as attribute-evaluation phrases (e.g., 'kuah sedap sangat', 'not too spicy') and three-word code-switching sequences.

In [6]:
print("=" * 70)
print("TOP 25 UNFILTERED TRIGRAMS")
print("=" * 70)
trigram_df = ngram_to_df(trigram_counts, trigram_reviews, total_unfiltered, 25, "trigram")
display(trigram_df)

print("\n" + "=" * 70)
print("TOP 25 FILTERED TRIGRAMS (stop words removed)")
print("=" * 70)
trigram_filt_df = ngram_to_df(trigram_filtered, trigram_filt_reviews, total_unfiltered, 25, "trigram")
display(trigram_filt_df)

TOP 25 UNFILTERED TRIGRAMS


,ngram,frequency,reviews_containing,per_1000_tokens,token_languages,cross_language
0,mi shin ramyun,6,6,2.664,Malay + brand + brand,False
1,shin ramyun punya,5,5,2.220,brand + brand + Malay,False
2,kenyal dan kuahnya,5,5,2.220,Malay + Malay + Malay,False
3,the noodles are,4,4,1.776,English + English + English,False
4,shin ramyun ni,4,4,1.776,brand + brand + Malay,False
5,kuah shin ramyun,4,4,1.776,Malay + brand + brand,False
6,shin ramyun is,3,3,1.332,brand + brand + English,False
7,of the noodles,3,3,1.332,English + English + English,False
8,i love this,3,3,1.332,English + English + English,False
9,shin ramyun sedap,3,3,1.332,brand + brand + Malay,False



TOP 25 FILTERED TRIGRAMS (stop words removed)


,ngram,frequency,reviews_containing,per_1000_tokens,token_languages,cross_language
0,mi shin ramyun,6,6,2.664,Malay + brand + brand,False
1,shin ramyun punya,5,5,2.220,brand + brand + Malay,False
2,kuah shin ramyun,4,4,1.776,Malay + brand + brand,False
3,shin ramyun sedap,4,4,1.776,brand + brand + Malay,False
4,shin ramyun sampai,3,3,1.332,brand + brand + ambiguous,False
5,not too spicy,2,2,0.888,ambiguous + ambiguous + English,False
6,too spicy but,2,2,0.888,ambiguous + English + ambiguous,False
7,flavour okay but,2,2,0.888,English + ambiguous + ambiguous,False
8,shin ramyun memang,2,2,0.888,brand + brand + Malay,False
9,shin ramyun noodles,2,2,0.888,brand + brand + English,False


## 7. Cross-language bigrams: code-switching at the word boundary

Cross-language bigrams are pairs where one token is classified as English and the other as Malay (or vice versa). These bigrams directly represent **intra-sentential code-switching at word boundaries** and are key evidence for understanding how reviewers mix languages when discussing product attributes.

Brand/product tokens and ambiguous words are excluded from cross-language classification to avoid false positives.

In [7]:
def get_cross_language_ngrams(counts, review_sets, total_tokens, top_n=30):
    """Extract n-grams where adjacent tokens are from different identified languages."""
    cross = []
    for gram, count in counts.most_common():
        langs = [get_token_language(t) for t in gram]
        real_langs = [l for l in langs if l not in ("brand", "ambiguous")]
        if len(real_langs) >= 2 and len(set(real_langs)) > 1:
            gram_str = " ".join(gram)
            lang_pair = " → ".join(langs)
            cross.append({
                "ngram": gram_str,
                "frequency": count,
                "reviews_containing": len(review_sets.get(gram, set())),
                "per_1000_tokens": round(count / total_tokens * 1000, 3) if total_tokens else 0,
                "language_direction": lang_pair,
            })
        if len(cross) >= top_n:
            break
    return pd.DataFrame(cross)

print("=" * 70)
print("TOP CROSS-LANGUAGE BIGRAMS (English ↔ Malay word boundaries)")
print("=" * 70)
cross_bigrams_df = get_cross_language_ngrams(bigram_counts, bigram_reviews, total_unfiltered, 30)
if len(cross_bigrams_df) > 0:
    display(cross_bigrams_df)
    print(f"\nTotal cross-language bigram types: {len(cross_bigrams_df)}")
else:
    print("No cross-language bigrams found with current classifications.")

print("\n" + "=" * 70)
print("TOP CROSS-LANGUAGE TRIGRAMS")
print("=" * 70)
cross_trigrams_df = get_cross_language_ngrams(trigram_counts, trigram_reviews, total_unfiltered, 20)
if len(cross_trigrams_df) > 0:
    display(cross_trigrams_df)
else:
    print("No cross-language trigrams found with current classifications.")

TOP CROSS-LANGUAGE BIGRAMS (English ↔ Malay word boundaries)


,ngram,frequency,reviews_containing,per_1000_tokens,language_direction
0,rasa kick,1,1,0.444,Malay → English
1,kick dgn,1,1,0.444,English → Malay
2,dgn spice,1,1,0.444,Malay → English
3,kari instant,1,1,0.444,Malay → English
4,noodle dalam,1,1,0.444,English → Malay
5,yg soup,1,1,0.444,Malay → English
6,soup mmg,1,1,0.444,English → Malay
7,flavour yang,1,1,0.444,English → Malay
8,kari has,1,1,0.444,Malay → English
9,kuah rich,1,1,0.444,Malay → English



Total cross-language bigram types: 28

TOP CROSS-LANGUAGE TRIGRAMS


,ngram,frequency,reviews_containing,per_1000_tokens,language_direction
0,rasa kick dgn,1,1,0.444,Malay → English → Malay
1,kick dgn spice,1,1,0.444,English → Malay → English
2,dengan kari instant,1,1,0.444,Malay → Malay → English
3,kari instant noodle,1,1,0.444,Malay → English → English
4,instant noodle dalam,1,1,0.444,English → English → Malay
5,noodle dalam malaysia,1,1,0.444,English → Malay → brand
6,this brand yg,1,1,0.444,English → ambiguous → Malay
7,flavour series sedap,1,1,0.444,English → ambiguous → Malay
8,or crabstick pon,1,1,0.444,English → ambiguous → Malay
9,yg soup mmg,1,1,0.444,Malay → English → Malay


## 8. N-gram patterns by review language label

Compare the most frequent content-word bigrams across English, Malay and code-switched reviews. These are inherited language labels from the source Sheet, not automatic detection. The comparison shows which attribute-related word pairs appear in each language group.

In [8]:
def ngrams_by_language(reviews, n, top_n=15):
    """Extract top filtered n-grams grouped by review language label."""
    lang_groups = {}
    for row in reviews:
        if row.get("included") is False:
            continue
        lang = row.get("language", "unknown")
        if lang not in lang_groups:
            lang_groups[lang] = Counter()
        for sentence in row.get("sentence_tokens_before", []):
            tokens = [t.casefold() for t in sentence if t.casefold() not in stopword_set]
            for gram in extract_ngrams(tokens, n):
                lang_groups[lang][gram] += 1
    
    results = {}
    for lang in sorted(lang_groups):
        rows = []
        for gram, count in lang_groups[lang].most_common(top_n):
            gram_str = " ".join(gram)
            token_langs = [get_token_language(t) for t in gram]
            rows.append({
                "ngram": gram_str,
                "frequency": count,
                "token_languages": " + ".join(token_langs),
            })
        results[lang] = pd.DataFrame(rows)
    return results

bigrams_by_lang = ngrams_by_language(reviews, 2, 15)

for lang, df in bigrams_by_lang.items():
    print(f"\n{'=' * 50}")
    print(f"TOP 15 FILTERED BIGRAMS IN {lang.upper()} REVIEWS")
    print(f"{'=' * 50}")
    display(df)


TOP 15 FILTERED BIGRAMS IN CODE-SWITCHED REVIEWS


,ngram,frequency,token_languages
0,shin ramyun,21,brand + brand
1,ramyun punya,5,brand + Malay
2,maggi kari,3,brand + Malay
3,mi shin,3,Malay + brand
4,sedap jugak,2,Malay + Malay
5,makan malam,2,Malay + ambiguous
6,tak perlu,2,Malay + Malay
7,kuah kari,2,Malay + Malay
8,bila lapar,2,Malay + ambiguous
9,asam laksa,2,ambiguous + ambiguous



TOP 15 FILTERED BIGRAMS IN ENGLISH REVIEWS


,ngram,frequency,token_languages
0,shin ramyun,19,brand + brand
1,instant noodles,6,English + English
2,personal favorite,2,ambiguous + English
3,but can't,2,ambiguous + ambiguous
4,but noodles,2,ambiguous + English
5,not so,2,ambiguous + ambiguous
6,too spicy,2,ambiguous + English
7,noodles stay,2,English + ambiguous
8,so much,2,ambiguous + ambiguous
9,not too,2,ambiguous + ambiguous



TOP 15 FILTERED BIGRAMS IN MALAY REVIEWS


,ngram,frequency,token_languages
0,shin ramyun,17,brand + brand
1,terlalu masin,3,ambiguous + Malay
2,mi shin,3,Malay + brand
3,kenyal kuahnya,3,Malay + Malay
4,mamee chef,2,ambiguous + ambiguous
5,tidak terlalu,2,ambiguous + ambiguous
6,terlalu pedas,2,ambiguous + Malay
7,kuahnya pekat,2,Malay + Malay
8,kenyal tak,2,Malay + Malay
9,terlalu lembik,2,ambiguous + Malay


## 9. Attribute-focused n-gram analysis

Group n-grams by the product attribute they discuss. This directly addresses the research question about which attributes are frequently discussed. The attribute grouping is based on the presence of known attribute terms.

In [9]:
ATTRIBUTE_TERMS = {
    "taste": {"taste", "flavour", "flavor", "rasa", "sedap", "enak", "lazat", "nikmat",
              "spicy", "pedas", "salty", "masin", "bland", "delicious"},
    "texture": {"texture", "springy", "chewy", "kenyal", "lembik", "firm", "soft",
                "soggy", "crispy", "rangup", "thick", "thin", "thinner"},
    "broth/soup": {"soup", "broth", "kuah", "kuahnya", "kari", "cair", "pekat", "rich"},
    "price/value": {"price", "harga", "mahal", "pricey", "worth", "value", "cheap", "affordable"},
    "packaging": {"packaging", "bungkusan", "packet", "portion", "sachet"},
    "preparation": {"cook", "cooking", "masak", "rebus", "goreng", "boil"},
}

def attribute_ngrams(counts, review_sets, total_tokens, n_gram_size=2):
    """Classify n-grams by which attribute category they relate to."""
    results = {attr: [] for attr in ATTRIBUTE_TERMS}
    for gram, count in counts.most_common():
        gram_tokens = set(gram)
        for attr, terms in ATTRIBUTE_TERMS.items():
            if gram_tokens & terms:
                langs = [get_token_language(t) for t in gram]
                real_langs = {l for l in langs if l not in ("brand", "ambiguous")}
                results[attr].append({
                    "ngram": " ".join(gram),
                    "frequency": count,
                    "reviews": len(review_sets.get(gram, set())),
                    "token_languages": " + ".join(langs),
                    "cross_language": len(real_langs) > 1,
                })
    return results

attr_bigrams = attribute_ngrams(bigram_filtered, bigram_filt_reviews, total_unfiltered)

print("ATTRIBUTE-RELATED BIGRAMS (filtered, sorted by frequency)")
print("=" * 70)
for attr in ATTRIBUTE_TERMS:
    items = attr_bigrams[attr][:10]  # top 10 per attribute
    if items:
        print(f"\n--- {attr.upper()} ---")
        df = pd.DataFrame(items)
        display(df)
    else:
        print(f"\n--- {attr.upper()} --- (no bigrams found)")

ATTRIBUTE-RELATED BIGRAMS (filtered, sorted by frequency)

--- TASTE ---


,ngram,frequency,reviews,token_languages,cross_language
0,ramyun sedap,4,4,brand + Malay,False
1,too spicy,3,3,ambiguous + English,False
2,terlalu masin,3,3,ambiguous + Malay,False
3,sedap jugak,2,2,Malay + Malay,False
4,spicy but,2,2,English + ambiguous,False
5,sangat sedap,2,2,Malay + Malay,False
6,terlalu pedas,2,2,ambiguous + Malay,False
7,sedap sangat,2,2,Malay + Malay,False
8,flavour okay,2,2,English + ambiguous,False
9,pedas but,2,2,Malay + ambiguous,False



--- TEXTURE ---


,ngram,frequency,reviews,token_languages,cross_language
0,kenyal kuahnya,5,5,Malay + Malay,False
1,texture noodles,2,2,English + English,False
2,kenyal tak,2,2,Malay + Malay,False
3,terlalu lembik,2,2,ambiguous + Malay,False
4,ramyun kenyal,2,2,brand + Malay,False
5,kekal kenyal,2,2,ambiguous + Malay,False
6,masih kenyal,2,2,Malay + Malay,False
7,firm noodles,2,2,English + English,False
8,noodles thinner,1,1,English + ambiguous,False
9,thinner other,1,1,ambiguous + ambiguous,False



--- BROTH/SOUP ---


,ngram,frequency,reviews,token_languages,cross_language
0,maggi kari,5,4,brand + Malay,False
1,kenyal kuahnya,5,5,Malay + Malay,False
2,kuah shin,4,4,Malay + brand,False
3,kuah kari,2,2,Malay + Malay,False
4,buat kuah,2,2,Malay + Malay,False
5,kuahnya pekat,2,2,Malay + Malay,False
6,but broth,2,2,ambiguous + English,False
7,kuahnya terlalu,2,2,Malay + ambiguous,False
8,supaya kuahnya,2,2,ambiguous + Malay,False
9,terlalu pekat,2,2,ambiguous + Malay,False


... (2 outputs left out)


--- PACKAGING ---


,ngram,frequency,reviews,token_languages,cross_language
0,one packet,4,4,ambiguous + English,False
1,comes sachet,1,1,ambiguous + ambiguous,False
2,sachet duck,1,1,ambiguous + ambiguous,False
3,generous portion,1,1,ambiguous + English,False
4,portion noodle,1,1,English + English,False
5,seasoning packet,1,1,English + English,False
6,packet tastes,1,1,English + ambiguous,False
7,packet feel,1,1,English + ambiguous,False
8,bungkusan shin,1,1,Malay + brand,False
9,packet filling,1,1,English + ambiguous,False



--- PREPARATION ---


,ngram,frequency,reviews,token_languages,cross_language
0,like cook,1,1,English + English,False
1,cook less,1,1,English + ambiguous,False
2,water boil,1,1,ambiguous + ambiguous,False
3,boil longer,1,1,ambiguous + ambiguous,False
4,boleh masak,1,1,Malay + Malay,False
5,masak biasa,1,1,Malay + ambiguous,False
6,perlu goreng,1,1,Malay + Malay,False
7,goreng just,1,1,Malay + ambiguous,False
8,just rebus,1,1,ambiguous + Malay,False
9,rebus celur,1,1,Malay + Malay,False


## 10. Bigram probabilities: MLE and Laplace smoothing

Sections 5–9 report raw counts. This section turns the cross-language bigrams into conditional probabilities, following the lecture's N-gram language model.

**Maximum likelihood estimate (MLE).** The probability of a word given the previous word is its bigram count divided by the count of the previous word:

P(wₙ | wₙ₋₁) = C(wₙ₋₁ wₙ) / C(wₙ₋₁)

**Laplace (add-one) smoothing.** MLE gives probability 0 to any bigram not seen in the corpus. With only 125 reviews, many valid word pairs are unseen, so one is added to every count. The denominator grows by V, the vocabulary size, because each of the V possible next words gains one count:

P*(wₙ | wₙ₋₁) = (C(wₙ₋₁ wₙ) + 1) / (C(wₙ₋₁) + V)

**Reconstituted count.** To see how much smoothing changes the original counts:

c*(wₙ₋₁ wₙ) = (C(wₙ₋₁ wₙ) + 1) × C(wₙ₋₁) / (C(wₙ₋₁) + V)

C(wₙ₋₁) is the unigram count of the previous word, and V is the number of distinct tokens in the unfiltered corpus. The table covers the top 15 cross-language bigrams, which are the English↔Malay switch points. The output also shows a zero-count pair, where MLE gives 0 and Laplace gives a small non-zero value.

In [10]:
unigram_counts = Counter()
for row in reviews:
    for sent in row.get("sentence_tokens_before", []):
        unigram_counts.update(t.casefold() for t in sent)

V = len(unigram_counts)  

def p_mle(w1, w2):
    """P(w2 | w1) = C(w1 w2) / C(w1)"""
    c1 = unigram_counts[w1]
    return bigram_counts[(w1, w2)] / c1 if c1 else 0.0

def p_laplace(w1, w2, k=1):
    """P*(w2 | w1) = (C(w1 w2) + k) / (C(w1) + k*V); k=1 is add-one"""
    return (bigram_counts[(w1, w2)] + k) / (unigram_counts[w1] + k * V)

def c_star(w1, w2):
    """Reconstituted count = (C(w1 w2) + 1) * C(w1) / (C(w1) + V)"""
    c1 = unigram_counts[w1]
    return (bigram_counts[(w1, w2)] + 1) * c1 / (c1 + V)

print(f"Vocabulary size V: {V:,}")

if len(cross_bigrams_df) == 0:
    print("No cross-language bigrams available; nothing to calculate.")
    prob_df = pd.DataFrame()
else:
    rows = []
    for _, r in cross_bigrams_df.head(15).iterrows():
        w1, w2 = r["ngram"].split()
        rows.append({
            "bigram": r["ngram"],
            "C(w1 w2)": bigram_counts[(w1, w2)],
            "C(w1)": unigram_counts[w1],
            "P_MLE": round(p_mle(w1, w2), 4),
            "P_Laplace": round(p_laplace(w1, w2), 6),
            "c_star": round(c_star(w1, w2), 3),
        })
    prob_df = pd.DataFrame(rows)
    display(prob_df)

    w1, w2 = "rasa", "spicy"
    print(f"\nUnseen pair: '{w1} {w2}'  (C(w1 w2) = 0)")
    print(f"  MLE:     {p_mle(w1, w2):.6f}")
    print(f"  Laplace: {p_laplace(w1, w2):.6f}")
    print(f"  Hand check: (0 + 1) / ({unigram_counts[w1]} + {V}) = {1 / (unigram_counts[w1] + V):.6f}")

Vocabulary size V: 740


,bigram,C(w1 w2),C(w1),P_MLE,P_Laplace,c_star
0,rasa kick,1,13,0.0769,0.002656,0.035
1,kick dgn,1,2,0.5000,0.002695,0.005
2,dgn spice,1,2,0.5000,0.002695,0.005
3,kari instant,1,12,0.0833,0.002660,0.032
4,noodle dalam,1,11,0.0909,0.002663,0.029
5,yg soup,1,3,0.3333,0.002692,0.008
6,soup mmg,1,7,0.1429,0.002677,0.019
7,flavour yang,1,16,0.0625,0.002646,0.042
8,kari has,1,12,0.0833,0.002660,0.032
9,kuah rich,1,21,0.0476,0.002628,0.055



Unseen pair: 'rasa spicy'  (C(w1 w2) = 0)
  MLE:     0.000000
  Laplace: 0.001328
  Hand check: (0 + 1) / (13 + 740) = 0.001328


## 11. Summary statistics and save outputs

Save all n-gram tables for the report and further analysis.

In [11]:
import hashlib

full_bigram_df = ngram_to_df(bigram_counts, bigram_reviews, total_unfiltered, 50)
full_bigram_df.to_csv(OUTPUT_DIR / "bigram_frequency_unfiltered.csv", index=False, encoding="utf-8-sig")

full_bigram_filt_df = ngram_to_df(bigram_filtered, bigram_filt_reviews, total_unfiltered, 50)
full_bigram_filt_df.to_csv(OUTPUT_DIR / "bigram_frequency_filtered.csv", index=False, encoding="utf-8-sig")

full_trigram_df = ngram_to_df(trigram_counts, trigram_reviews, total_unfiltered, 50, "trigram")
full_trigram_df.to_csv(OUTPUT_DIR / "trigram_frequency_unfiltered.csv", index=False, encoding="utf-8-sig")

full_trigram_filt_df = ngram_to_df(trigram_filtered, trigram_filt_reviews, total_unfiltered, 50, "trigram")
full_trigram_filt_df.to_csv(OUTPUT_DIR / "trigram_frequency_filtered.csv", index=False, encoding="utf-8-sig")

all_cross_bigrams = get_cross_language_ngrams(bigram_counts, bigram_reviews, total_unfiltered, 100)
all_cross_bigrams.to_csv(OUTPUT_DIR / "cross_language_bigrams.csv", index=False, encoding="utf-8-sig")

attr_rows = []
for attr, items in attr_bigrams.items():
    for item in items[:15]:  
        item["attribute"] = attr
        attr_rows.append(item)
pd.DataFrame(attr_rows).to_csv(OUTPUT_DIR / "attribute_bigrams.csv", index=False, encoding="utf-8-sig")

lang_rows = []
for lang, df in bigrams_by_lang.items():
    for _, row in df.iterrows():
        lang_rows.append({"review_language": lang, **row.to_dict()})
pd.DataFrame(lang_rows).to_csv(OUTPUT_DIR / "bigrams_by_language.csv", index=False, encoding="utf-8-sig")

if "prob_df" in globals() and len(prob_df) > 0:
    prob_df.to_csv(OUTPUT_DIR / "bigram_probabilities.csv", index=False, encoding="utf-8-sig")
else:
    print("Run the bigram probability section first to create bigram_probabilities.csv")

inputs = [PROCESSED_DIR / f"{origin}_preprocessed.jsonl" for origin in ("real", "ai")]
inputs.append(PROCESSED_DIR / "stopwords_used.csv")
manifest = {
    "review_count": len(reviews),
    "unfiltered_tokens": total_unfiltered,
    "unfiltered_bigram_types": len(bigram_counts),
    "unfiltered_trigram_types": len(trigram_counts),
    "filtered_bigram_types": len(bigram_filtered),
    "filtered_trigram_types": len(trigram_filtered),
    "cross_language_bigram_types": len(all_cross_bigrams),
    "smoothing_vocabulary_size": V if "V" in globals() else None,
    "attribute_categories": list(ATTRIBUTE_TERMS.keys()),
    "sentence_bounded": True,
    "input_sha256": {
        str(p.relative_to(PROJECT_DIR)).replace("\\", "/"): hashlib.sha256(p.read_bytes()).hexdigest()
        for p in inputs
    },
}
(OUTPUT_DIR / "run_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

print("\n" + "=" * 70)
print("N-GRAM ANALYSIS SUMMARY")
print("=" * 70)
print(f"Reviews analysed: {len(reviews)}")
print(f"Unfiltered tokens: {total_unfiltered:,}")
print(f"Unfiltered bigram types: {len(bigram_counts):,}")
print(f"Unfiltered trigram types: {len(trigram_counts):,}")
print(f"Filtered bigram types: {len(bigram_filtered):,}")
print(f"Filtered trigram types: {len(trigram_filtered):,}")
print(f"Cross-language bigram types: {len(all_cross_bigrams)}")
if "V" in globals():
    print(f"Vocabulary size V (smoothing): {V:,}")
print(f"\nSaved files in {OUTPUT_DIR}:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")


N-GRAM ANALYSIS SUMMARY
Reviews analysed: 125
Unfiltered tokens: 2,252
Unfiltered bigram types: 1,666
Unfiltered trigram types: 1,761
Filtered bigram types: 1,259
Filtered trigram types: 1,185
Cross-language bigram types: 28
Vocabulary size V (smoothing): 740

Saved files in c:\Users\Admin\OneDrive\Documents\GitHub\CPC353-Assignment-1\data\analysis\ngram:
  attribute_bigrams.csv
  bigram_frequency_filtered.csv
  bigram_frequency_unfiltered.csv
  bigram_probabilities.csv
  bigrams_by_language.csv
  cross_language_bigrams.csv
  run_manifest.json
  trigram_frequency_filtered.csv
  trigram_frequency_unfiltered.csv


## 12. Interpretation and limits

The notebook creates:

- attribute_bigrams.csv
- bigram_frequency_filtered.csv
- bigram_frequency_unfiltered.csv
- bigram_probabilities.csv
- bigrams_by_language.csv
- cross_language_bigrams.csv
- run_manifest.json
- trigram_frequency_filtered.csv
- trigram_frequency_unfiltered.csv

The results combine the whole dataset across all 125 reviews. Use frequency and reviews_containing together to distinguish recurrent phrases from one-off expressions. Use `attribute_bigrams.csv` to report which attributes (taste, texture, broth) are most frequently discussed, and `cross_language_bigrams.csv` to evidence intra-sentential code-switching at word boundaries. Use `bigram_probabilities.csv` to compare MLE and Laplace-smoothed probabilities of the switch-point bigrams; smoothing gives unseen word pairs a small non-zero probability. The language-grouped tables show stylistic differences across English, Malay, and code-switched reviews without claiming automated language identification. Keep the corpus's combined real and synthetic origin clear in the report prose, and verify individual phrases in context using the full reviews. These are CSV tables that Excel can open; they are not Excel workbooks.

**Limitations:**
- Language tags at the token level come from a project-written lookup and not from automatic language detection. Ambiguous tokens are not counted in the cross-language statistics, so switch counts may be underestimated.
- N-grams are extracted within sentence boundaries as estimated; cross-sentence n-grams are not considered.
- Brand/product name tokens are excluded from language tagging but included in n-grams.
- The corpus is a convenience sample of collected and AI-generated reviews; n-gram frequencies do not reflect patterns across the whole category.
- Frequency does not imply meaning or sentiment. N-grams need to be interpreted with the context of the concordances and original reviews.
- With only 125 reviews, bigram probabilities rest on small counts, and add-one smoothing noticeably shifts them. Treat them as illustrative, not as a trained language model.

Method reference: D. Sarkar, *Text Analytics with Python: A Practitioner's Guide to Exploring and Analyzing Text Data*, 2nd ed., Apress, 2019, chapter 5, "Text Summarization" (Collocations and N-grams). This project implements its own native sliding-window routine (`extract_ngrams`) and `collections.Counter` within sentence boundaries to track review IDs, provenance, and cross-language boundaries; it does not call NLTK's `BigramCollocationFinder` on a flattened corpus.